<a href="https://colab.research.google.com/github/w3aarush/DR_Classification_NIT_MCA_Project/blob/main/ConvNextBase_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
# from tensorflow.keras.layers.experimental import preprocessing
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Attention
# from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
import matplotlib.pyplot as plt
################################################################################
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S, preprocess_input
from google.colab.patches import cv2_imshow
import pandas as pd
import numpy as np
import seaborn as sns
# import imutils
import time
import cv2
import os
# from cuml import SVC # for python 3.11
# from sklearn.svm import SVC

In [ ]:
# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()

# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download DATASET (not competition)
!kaggle datasets download -d mariaherrerot/aptos2019

# Unzip
!unzip -q aptos2019.zip -d aptos2019

# Check files
!ls aptos2019

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mariaherrerot/aptos2019
License(s): unknown
100% 8.01G/8.01G [01:20<00:00, 106MB/s]

test.csv  test_images  train_1.csv  train_images  valid.csv  val_images


In [ ]:
base_dir = '/content/aptos2019'
train_dir = '/content/aptos2019/train_images/train_images/'
validation_dir = '/content/aptos2019/val_images/val_images/'
test_dir = '/content/aptos2019/test_images/test_images/'

In [ ]:
print(os.listdir(train_dir))
print(os.listdir(validation_dir))
print(os.listdir(test_dir))

['291e2ff3d834.png', '8cb6b5b2f19c.png', 'a30a143a53a3.png', 'c25e02b39c01.png', '495255c7492f.png', 'b842b43cb7fb.png', 'd1a60c3b9fe5.png', '49419f8d5cb4.png', '5eb311bcb5f9.png', '2776d70724d3.png', '668e853258cd.png', '8f10e41a2f02.png', '1dfbede13143.png', 'a15652b22ab8.png', '7c3747c0b2c3.png', '306c841af3fc.png', '941d874c8afb.png', 'bc23f74e14dd.png', '7743f4e04a6d.png', '7a46cfa69bae.png', 'a3957df90a78.png', 'cb02bb47fdc5.png', '44e951e45dca.png', 'a21b37719f9b.png', '8c0d05233238.png', '6b91e99c9408.png', 'c639d837f5e4.png', '66cd9c28e636.png', 'ccd34029493d.png', '94b1d8ad35ec.png', '3a1ecf5e2839.png', '354b8911d6ed.png', '2da82d14e1b7.png', '709784f7fcc2.png', '6e0f78e188ff.png', 'a56230242a95.png', '57f933d3d7c7.png', '519c6e8f78dc.png', '3b5dffe159b6.png', '8dafa62f9322.png', '4cfd22ae43d4.png', '1d11794057ff.png', '3461dc601cc2.png', '5bda2ed09e62.png', 'bac1744955c2.png', '4350a1b2f3cb.png', '898f0bc8acfa.png', '3286073a976e.png', '1db18bdd43aa.png', 'a0adbe677508.png',

In [ ]:
NUM_CLASSES = 5
epochs = 20

In [ ]:
BATCH_SIZE = 32

In [ ]:
def crop_image_from_gray(img, tol=7):
    """
    Applies masks to the orignal image and
    returns the a preprocessed image with
    3 channels

    :param img: A NumPy Array that will be cropped
    :param tol: The tolerance used for masking

    :return: A NumPy array containing the cropped image
    """
    # If for some reason we only have two channels
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1),mask.any(0))]
    # If we have a normal RGB images
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol

        check_shape = img[:,:,0][np.ix_(mask.any(1),mask.any(0))].shape[0]
        if (check_shape == 0): # image is too dark so that we crop out everything,
            return img # return original image
        else:
            img1=img[:,:,0][np.ix_(mask.any(1),mask.any(0))]
            img2=img[:,:,1][np.ix_(mask.any(1),mask.any(0))]
            img3=img[:,:,2][np.ix_(mask.any(1),mask.any(0))]
            img = np.stack([img1,img2,img3],axis=-1)
        return img

# Define IMG_WIDTH and IMG_HEIGHT based on IMG_SIZE
# IMG_SIZE is already defined in a previous cell.
IMG_SIZE = 224
IMG_WIDTH = IMG_SIZE
IMG_HEIGHT = IMG_SIZE

def preprocess_image(image, sigmaX=10):
    """
    The whole preprocessing pipeline:
    1. Read in image
    2. Apply masks
    3. Resize image to desired size
    4. Add Gaussian noise to increase Robustness

    :param img: A NumPy Array that will be cropped
    :param sigmaX: Value used for add GaussianBlur to the image

    :return: A NumPy array containing the preprocessed image
    """
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = crop_image_from_gray(image)
    image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
    image = cv2.addWeighted (image,4, cv2.GaussianBlur(image, (0,0) ,sigmaX), -4, 128)
    return image

In [ ]:
def tf_preprocess_image(image_path, label, sigmaX):
    image_path_str = image_path.numpy().decode('utf-8')
    # Ensure sigmaX is a plain Python int or float for OpenCV
    sigmaX_val = sigmaX.numpy().item()

    image = preprocess_image(cv2.imread(image_path_str), sigmaX=sigmaX_val)
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    # Normalize the image pixels to the range [0, 1]
    image = image / 255.0

    # Softmax label is the original label, cast to int32
    softmax_label = tf.cast(label, tf.int32)

    # CORN labels conversion
    corn_labels = np.zeros(num_classes - 1, dtype=np.int32) # num_classes is global
    for k in range(num_classes - 1):
        if label.numpy() > k:
            corn_labels[k] = 1
    corn_labels = tf.convert_to_tensor(corn_labels, dtype=tf.int32)

    return image, softmax_label, corn_labels

def load_and_preprocess_tf(image_path, label, sigmaX):
    # Wrap the python function for tf.data
    image, softmax_label_tensor, corn_labels_tensor = tf.py_function(
        tf_preprocess_image, [image_path, label, sigmaX],
        [tf.float32, tf.int32, tf.int32] # Specify output dtypes for image, softmax_label, corn_labels
    )
    # Reconstruct the dictionary for labels
    labels_dict = {'softmax_head': softmax_label_tensor, 'corn_head': corn_labels_tensor}

    image.set_shape([IMG_SIZE, IMG_SIZE, 3]) # Set shape for batching
    labels_dict['softmax_head'].set_shape([]) # Set shape for batching (scalar)
    labels_dict['corn_head'].set_shape([num_classes - 1]) # Set shape for batching (vector of length num_classes - 1)

    return image, labels_dict

In [ ]:
from tensorflow.keras import layers

img_augmentation = Sequential(
    [
        layers.InputLayer(input_shape=(IMG_SIZE, IMG_SIZE, 3)), # Explicitly define input shape
        tf.keras.layers.RandomRotation(factor=(-0.15, 0.15)),
        tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        tf.keras.layers.RandomFlip(),
        tf.keras.layers.RandomContrast(factor=0.1),
    ],
    name="img_augmentation",
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [ ]:
num_classes = 5
from tensorflow.keras.applications import ConvNeXtBase

def build_ConvNextBase(num_classes):
    IMG_SIZE = 224
    IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3) # Using the IMG_SIZE defined earlier, and 3 for RGB channels

    # Load the ConvNeXtBase model without the top classification layer
    base_model = ConvNeXtBase(weights='imagenet', include_top=False, input_shape=IMG_SHAPE)

    # freeze the base model
    base_model.trainable = False

    # create a new model on the top of the model
    inputs = keras.Input(shape=IMG_SHAPE)
    # Preprocessing with `preprocess_image` should be done in the tf.data.Dataset pipeline
    x = img_augmentation(inputs) # Apply data augmentation
    x = tf.keras.applications.convnext.preprocess_input(x) # ConvNeXt specific preprocessing
    x = base_model(x, training=False)
    shared_features = layers.GlobalAveragePooling2D()(x)
    shared_features = layers.Dense(100, activation='relu')(shared_features)
    shared_features = layers.Dropout(0.5)(shared_features)

    # Softmax Head
    softmax_output = layers.Dense(num_classes, activation='softmax', name='softmax_head')(shared_features)

    # CORN Head (Conditional Ordinal Regression)
    # It predicts num_classes - 1 cumulative probabilities P(y > k)
    corn_output = layers.Dense(num_classes - 1, activation='sigmoid', name='corn_head')(shared_features)

    model = keras.Model(inputs=inputs, outputs=[softmax_output, corn_output])
    # Explicitly build the model to ensure all variables are created
    model.build(input_shape=tf.TensorShape([None] + list(IMG_SHAPE)))

    # Compile the model
    model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'softmax_head': keras.losses.SparseCategoricalCrossentropy(),
        'corn_head': keras.losses.BinaryCrossentropy() # Placeholder: A custom CORN loss function is typically used here.
    },
    metrics={
        'softmax_head': ['accuracy'],
        'corn_head': ['accuracy'] # Placeholder: More appropriate metrics for CORN might be needed.
    }
    )
    return model

### Create `tf.data.Dataset` objects

In [ ]:
def load_data():
    train = pd.read_csv('/content/aptos2019/train_1.csv', encoding='utf-8')
    test = pd.read_csv('/content/aptos2019/test.csv', encoding='utf-8')
    valid = pd.read_csv('/content/aptos2019/valid.csv')

    train_dir = '/content/aptos2019/train_images/train_images/'
    test_dir = '/content/aptos2019/test_images/test_images/'
    valid_dir = '/content/aptos2019/val_images/val_images/'

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    # Remove redundant columns
    # train['train_images'] = train['id_code'] + '.png'
    # test['test_images'] = test['id_code'] + '.png'
    # valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(int) # Change to int for SparseCategoricalCrossentropy
    test['diagnosis'] = test['diagnosis'].astype(int)   # Change to int
    valid['diagnosis'] = valid['diagnosis'].astype(int) # Change to int

    return train, test, valid, train_dir, valid_dir, test_dir

In [ ]:
# Load dataframes
train_df, test_df, valid_df, train_img_dir, valid_img_dir, test_img_dir = load_data()

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_df['image_path'].values, train_df['diagnosis'].values))
valid_dataset = tf.data.Dataset.from_tensor_slices((valid_df['image_path'].values, valid_df['diagnosis'].values))
test_dataset = tf.data.Dataset.from_tensor_slices((test_df['image_path'].values, test_df['diagnosis'].values))

# Apply preprocessing
train_dataset = train_dataset.map(lambda x, y: load_and_preprocess_tf(x, y, sigmaX=30), num_parallel_calls=tf.data.AUTOTUNE)
valid_dataset = valid_dataset.map(lambda x, y: load_and_preprocess_tf(x, y, sigmaX=30), num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.map(lambda x, y: load_and_preprocess_tf(x, y, sigmaX=30), num_parallel_calls=tf.data.AUTOTUNE)

# Configure for performance
BATCH_SIZE = 32
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
valid_dataset = valid_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
if __name__ == "__main__":
    model = build_ConvNextBase(num_classes)

    # Callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4, verbose=1)

    history = model.fit(train_dataset, epochs=20, validation_data=valid_dataset, callbacks=[early_stopping, reduce_lr])

350926856/350926856 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 899s 9s/step - corn_head_accuracy: 0.4416 - corn_head_loss: 0.6959 - loss: 2.2485 - softmax_head_accuracy: 0.4334 - softmax_head_loss: 1.5504 - val_corn_head_accuracy: 0.7213 - val_corn_head_loss: 0.5323 - val_loss: 1.7958 - val_softmax_head_accuracy: 0.4699 - val_softmax_head_loss: 1.2823 - learning_rate: 1.0000e-04
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 825s 9s/step - corn_head_accuracy: 0.5662 - corn_head_loss: 0.5533 - loss: 1.7767 - softmax_head_accuracy: 0.5505 - softmax_head_loss: 1.2230 - val_corn_head_accuracy: 0.9044 - val_corn_head_loss: 0.5178 - val_loss: 1.7098 - val_softmax_head_accuracy: 0.5246 - val_softmax_head_loss: 1.2094 - learning_rate: 1.0000e-04
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 916s 10s/step - corn_head_accuracy: 0.6519 - corn_head_loss: 0.5123 - loss: 1.6399 - softmax_head_accuracy: 0.6014 - softmax_head_loss: 1.1274 - val_corn_head_accuracy: 0.9781 - val_corn_head_los

In [ ]:
model.save('convnextbase.keras')

In [ ]:
model = build_ConvNextBase(num_classes)

In [ ]:
model.summary()

In [ ]:
model.summary()